# Inference for CLARIS

This notebook contains the code required for inference from CLARIS.
It is preferred to use the `infer_beam.py` & `generate_waveform_from_code.py` scripts for inference but this notebook serves as a simple way to follow along and understand the code.

First, we import necessary packages and define some constants

In [ ]:
# System-packages
import os
import torch
import soundfile as sf

# Code files
from model import SpeechToUnitTransformer
from params import ModelParams, CodeGeneratorParams
from data.dataset import getFeaturesOrWaveform
from sequence_generator import SequenceGenerator
from utils import (
    Dictionary,
    getSymbolsToStripFromOutput,
    getModelStateDictFromPath,
    postProcessPrediction,
    stripPad,
)
from vocoders.code_hifigan import CodeHiFiGANVocoder
import IPython.display as ipd

# Modify these paths as you require
CLARIS_CHECKPOINT_PATH = "./checkpoint.pt"
VOCODER_CHECKPOINT_PATH = "./code_hifigan.pt"
INPUT_AUDIO_PATH = "./whisper.wav"

### Initializing the Speech-to-unit model

In [ ]:
tgtDict = Dictionary()
for i in range(1000):
    # 1000 -> target code size, can be changed for new models 
    tgtDict.addSymbol(str(i))
    
# Building the model
modelParams = ModelParams()
model = SpeechToUnitTransformer.buildModel(
    params = modelParams,
    tgtDict = tgtDict,
)

# Now, we load the checkpoint
modelStDict = model.state_dict()
model.load_state_dict(getModelStateDictFromPath(
    CLARIS_CHECKPOINT_PATH,
    modelStDict,
))
model.eval()

# The generator handles the BeamSearch logic
generator = SequenceGenerator(
    model = model,
    tgtDict = tgtDict,
    beamSize = 20,
    maxLenA = 1,
    maxLen = 50_000,
)

### Initializing the Vocoder

In [ ]:
vocoderParams = CodeGeneratorParams()
vocoder = CodeHiFiGANVocoder(
    VOCODER_CHECKPOINT_PATH,
    vocoderParams
)

Now that the models are loaded, we load the audio file to serve as input to the model.

In [ ]:
audioFeaturesNp = getFeaturesOrWaveform(
    path = INPUT_AUDIO_PATH
)

# audioFeatures is a numpy array, we need to convert it to torch Tensor before inference
audioFeatures: torch.Tensor = torch.from_numpy(audioFeaturesNp).float()

The `getFeaturesOrWaveform` method uses `fbank` for `torchaudio.ta_kaldi` to provide the mel-spectrograms for input audio.

We now have all necessary objects and data to start inference. Inference starts through the `SequenceGenerator` which uses beam search to auto-regressively generate units.

In [ ]:
hypos = generator.generate(
    srcTokens = audioFeatures.unsqueeze(0), # Simulate batching by adding an extra dimension
    srcLengths = torch.tensor([audioFeatures.size(0)], dtype = torch.long), # Number of frames in time dimension
    ids = torch.tensor([0], dtype = torch.long) # Dummy id passed, won't affect inference in any way
)

`hypos` now contains all beams representing the hypotheses by the model.

We need only pre-process it a little (remove the special tokens and re-adjust the remaining tokens so that it is a valid sequence for the vocoder). The `postProcessPrediction` method will help us with this task.

In [ ]:
for i, hypo in enumerate(hypos[0][:modelParams.generation.nbest]):
    # This loop can in theory select multiple beams at once however, modelParams define nBest as 1 
    # which means that the best beam will be selected
    hypoTokens, hypoStr = postProcessPrediction(
        hypoTokens = hypo['tokens'].int().cpu(),
        srcStr = "",
        alignment = hypo['alignment'],
        alignDict = None,
        tgtDict = tgtDict,
        removeBpe = None,
        extraSymbolsToIgnore = getSymbolsToStripFromOutput(generator),
    )
    
    # hypoTokens contains the model predictions
    # hypoStr is what we are interested in. This is a string that contains the units that we want to pass to the vocoder
    # Since it is a string, we first need to convert it into a list of ints
    units = [int(unit) for unit in hypoStr.split()]

In [ ]:
# We can print both hypoTokens and hypoStr to observe that they are essentially the same just shifted by 4.
# This is because our dictionary object adds 4 special tokens at the initial indices during creation.
# This causes a shift of 4 for all units and will cause the vocoder to fail if hypoTokens is passed directly

print(f"hypoTokens:\n{hypoTokens}")
print("\n\n")
print(f"hypoStr:\n{hypoStr}")

We now have the units. The vocoder can use these to generate clear speech

In [ ]:
codeTensor = torch.LongTensor(units).view(1, -1) # Converting the list of int units to torch Tensor
wav = vocoder(code = codeTensor, durPrediction = True)

# The audio is generated, we display it in the notebook using IPython.Display
ipd.display(ipd.Audio(wav, rate=16_000, normalize=False))

For inference using ONNX or OpenVino checkpoints, the `SequenceGenerator` will need to be updated to use the newer checkpoints and manage ONNXRuntimes properly